# Targeted Attention Head Ablation on GPT-2 (Small)

This notebook identifies and neutralizes "knowledge conduits" — attention heads
that allocate the most weight to a subject token when the model predicts a factual
target — then measures how factual recall degrades under ablation.

## 1. Imports

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings("ignore")

print("Imports OK  |  CUDA available:", torch.cuda.is_available())

## 2. Load Model, Tokenizer & Data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2", output_attentions=True)
model.to(device)
model.eval()

NUM_LAYERS = model.config.n_layer   # 12
NUM_HEADS  = model.config.n_head    # 12
print(f"GPT-2 small loaded: {NUM_LAYERS} layers × {NUM_HEADS} heads  |  device={device}")

In [ ]:
df = pd.read_csv("ex1_data.csv")
print(f"Dataset: {len(df)} rows")
df.head()

## 3. Utility — Find the Last Sub-word Token Index of the Subject

GPT-2 uses BPE sub-word tokenization (e.g. `"Potter"` → `["Pot", "ter"]`).
This function robustly locates the **last** sub-word token belonging to the
subject within the tokenized prompt, using sliding-window matching with a
character-level fallback.

In [ ]:
def find_subject_last_token_index(prompt: str, subject: str, tokenizer: GPT2Tokenizer) -> int:
    """
    Robustly locate the index (within the tokenized prompt) of the *last*
    sub-word token that belongs to `subject`.

    Strategy:
      1. Tokenize the full prompt → token_ids.
      2. Tokenize the subject alone → subj_ids.
      3. Slide a window of len(subj_ids) over token_ids and look for a match.
         Because GPT-2's BPE can produce different tokenizations depending on
         leading whitespace, we also try the subject prefixed with a space.
      4. If no exact match is found, fall back to a decoded-string character
         offset search (handles edge-case BPE splits).

    Returns the 0-based position of the last subject token in the prompt's
    token list, or -1 if the subject cannot be found.
    """
    prompt_ids = tokenizer.encode(prompt)

    # Try both with and without leading space (GPT-2 BPE is space-sensitive)
    for variant in [subject, " " + subject]:
        subj_ids = tokenizer.encode(variant)
        window = len(subj_ids)
        for start in range(len(prompt_ids) - window + 1):
            if prompt_ids[start : start + window] == subj_ids:
                return start + window - 1  # last sub-word of subject

    # ── Fallback: character-level alignment ──────────────────────────────────
    decoded_tokens = [tokenizer.decode([tid]) for tid in prompt_ids]
    reconstructed = ""
    spans = []
    for tok_str in decoded_tokens:
        start_c = len(reconstructed)
        reconstructed += tok_str
        spans.append((start_c, len(reconstructed)))

    # Find last occurrence of subject in the reconstructed string
    idx = reconstructed.rfind(subject)
    if idx == -1:
        idx = reconstructed.lower().rfind(subject.lower())
    if idx == -1:
        return -1

    subj_end_char = idx + len(subject)
    for i, (s, e) in enumerate(spans):
        if s <= subj_end_char - 1 < e:
            return i
    return -1

### Sanity Check

In [ ]:
test_prompt = "The author of Harry Potter is"
test_subj   = "Potter"
idx = find_subject_last_token_index(test_prompt, test_subj, tokenizer)
test_ids = tokenizer.encode(test_prompt)
print(f"Prompt tokens : {[tokenizer.decode([t]) for t in test_ids]}")
print(f"Subject '{test_subj}' → last token index = {idx}  "
      f"('{tokenizer.decode([test_ids[idx]])}')")

## 4. Mapping — Extract Per-Head Attention on the Subject Token

For each prompt we run a forward pass and record, for every `(layer, head)`,
the attention weight that the **last position** (the prediction site) places
on the last subject token.

In [ ]:
@torch.no_grad()
def get_attention_to_subject(
    prompt: str,
    subject: str,
    model: GPT2LMHeadModel,
    tokenizer: GPT2Tokenizer,
    device: torch.device,
) -> Tuple[np.ndarray, int]:
    """
    Returns
    -------
    attn_matrix : np.ndarray, shape (NUM_LAYERS, NUM_HEADS)
        attn_matrix[l, h] = attention weight from last position to subject
        token in layer l, head h.
    subj_idx : int
        Token index of the last subject sub-word.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    subj_idx = find_subject_last_token_index(prompt, subject, tokenizer)
    assert subj_idx != -1, f"Subject '{subject}' not found in prompt '{prompt}'"

    outputs = model(input_ids, output_attentions=True)
    attentions = outputs.attentions  # tuple of (1, num_heads, seq_len, seq_len)

    last_pos = input_ids.shape[1] - 1  # the position that predicts the target

    attn_matrix = np.zeros((NUM_LAYERS, NUM_HEADS))
    for layer_i, attn in enumerate(attentions):
        for head_j in range(NUM_HEADS):
            attn_matrix[layer_i, head_j] = (
                attn[0, head_j, last_pos, subj_idx].item()
            )
    return attn_matrix, subj_idx

print("get_attention_to_subject() defined.")

## 5. Intervention — Hook-Based Head Ablation

We use `register_forward_hook` to intercept the attention module's output and
zero out the contribution of specific heads to the residual stream — all
**without** modifying the model's actual weights.

### 5a. Ablation Hook Factory

In [ ]:
def make_ablation_hook(head_indices_to_zero: List[int]):
    """
    Return a forward-hook function that zeros out the output of specific
    attention heads inside a GPT2Attention module.

    In GPT-2 (Hugging Face), each transformer block's attention module
    (`GPT2Attention`) returns a tuple:
        (attn_output, present, attn_weights)
    where `attn_output` has shape (batch, seq_len, hidden_dim).

    hidden_dim = num_heads × head_dim.  We reshape, zero the target
    heads, then reshape back — all without touching model weights.
    """
    def hook_fn(module, input, output):
        attn_output = output[0]  # (batch, seq_len, hidden_dim)
        batch, seq_len, hidden = attn_output.shape
        head_dim = hidden // NUM_HEADS

        # Reshape to (batch, seq_len, num_heads, head_dim)
        reshaped = attn_output.view(batch, seq_len, NUM_HEADS, head_dim)

        # Zero out selected heads
        for h in head_indices_to_zero:
            reshaped[:, :, h, :] = 0.0

        modified_output = reshaped.view(batch, seq_len, hidden)
        return (modified_output,) + output[1:]

    return hook_fn

print("make_ablation_hook() defined.")

### 5b. Target Token Probability (with optional ablation)

In [ ]:
@torch.no_grad()
def get_target_prob(
    prompt: str,
    target_token: str,
    model: GPT2LMHeadModel,
    tokenizer: GPT2Tokenizer,
    device: torch.device,
    ablation_spec: Optional[Dict[int, List[int]]] = None,
) -> float:
    """
    Compute the probability the model assigns to `target_token` as the next
    token after `prompt`.

    Parameters
    ----------
    ablation_spec : dict mapping layer_index → list of head indices to zero.
        If None, runs standard (baseline) inference.

    Returns
    -------
    prob : float   (softmax probability of the target token)
    """
    handles = []
    if ablation_spec is not None:
        for layer_idx, head_list in ablation_spec.items():
            attn_module = model.transformer.h[layer_idx].attn
            hook = make_ablation_hook(head_list)
            handles.append(attn_module.register_forward_hook(hook))

    try:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        outputs = model(input_ids)
        logits = outputs.logits[0, -1, :]  # logits at last position
        probs = torch.softmax(logits, dim=-1)

        # Encode target with leading space (GPT-2 convention for word-initial tokens)
        target_id = tokenizer.encode(" " + target_token)[0]
        prob = probs[target_id].item()
    finally:
        for h in handles:
            h.remove()

    return prob

print("get_target_prob() defined.")

## 6. Build Ablation Specs from the Attention Matrix

Translate the `(layers × heads)` attention matrix into three ablation
specifications:

| Condition | Rule |
|-----------|------|
| **A** | Zero out the single head with the highest attention on the subject |
| **B** | Zero out the top-3 heads |
| **C** | Zero out all heads with attention weight > 0.25 |

In [ ]:
def build_ablation_specs(
    attn_matrix: np.ndarray,
    threshold: float = 0.25,
) -> Tuple[Dict[int, List[int]], Dict[int, List[int]], Dict[int, List[int]], Tuple[int, int]]:
    """
    Returns
    -------
    spec_a : dict  – Condition A (top-1 head zeroed)
    spec_b : dict  – Condition B (top-3 heads zeroed)
    spec_c : dict  – Condition C (all heads with attn > threshold)
    top1   : (layer, head) of the single most influential head
    """
    flat = attn_matrix.flatten()

    # ── Condition A: top-1 head ──────────────────────────────────────────────
    top1_flat = np.argmax(flat)
    top1_layer, top1_head = divmod(int(top1_flat), NUM_HEADS)
    spec_a: Dict[int, List[int]] = {top1_layer: [top1_head]}

    # ── Condition B: top-3 heads ─────────────────────────────────────────────
    top3_flat = np.argsort(flat)[-3:][::-1]
    spec_b: Dict[int, List[int]] = {}
    for idx in top3_flat:
        l, h = divmod(int(idx), NUM_HEADS)
        spec_b.setdefault(l, []).append(h)

    # ── Condition C: all heads above threshold ───────────────────────────────
    spec_c: Dict[int, List[int]] = {}
    above = np.argwhere(attn_matrix > threshold)
    for l, h in above:
        spec_c.setdefault(int(l), []).append(int(h))

    return spec_a, spec_b, spec_c, (top1_layer, top1_head)

print("build_ablation_specs() defined.")

## 7. Main Experiment Loop

For every prompt in the dataset:
1. **Map** — extract the attention-to-subject matrix.
2. **Build** ablation specs for Conditions A / B / C.
3. **Evaluate** baseline and intervened probabilities.
4. **Compute** relative probability decrease:

$$\Delta = \frac{p_{\text{baseline}} - p_{\text{intervention}}}{p_{\text{baseline}}}$$

In [ ]:
results = []

for row_idx, row in df.iterrows():
    prompt       = row["Prompt"]
    subject      = row["Subject Word(s)"]
    target_token = row["Target Token"]

    # 1. Mapping — attention to subject at the prediction site
    try:
        attn_matrix, subj_token_idx = get_attention_to_subject(
            prompt, subject, model, tokenizer, device
        )
    except AssertionError as e:
        print(f"[SKIP] Row {row_idx}: {e}")
        continue

    # 2. Build ablation specs for the three conditions
    spec_a, spec_b, spec_c, (top1_layer, top1_head) = build_ablation_specs(attn_matrix)

    # 3. Evaluate probabilities
    p_baseline = get_target_prob(prompt, target_token, model, tokenizer, device)
    p_cond_a   = get_target_prob(prompt, target_token, model, tokenizer, device, spec_a)
    p_cond_b   = get_target_prob(prompt, target_token, model, tokenizer, device, spec_b)
    p_cond_c   = get_target_prob(prompt, target_token, model, tokenizer, device, spec_c)

    # 4. Compute relative probability decrease
    eps = 1e-12
    delta_a = (p_baseline - p_cond_a) / max(p_baseline, eps)
    delta_b = (p_baseline - p_cond_b) / max(p_baseline, eps)
    delta_c = (p_baseline - p_cond_c) / max(p_baseline, eps)

    results.append({
        "prompt_id":        row_idx,
        "baseline_prob":    round(p_baseline, 6),
        "condition_a_delta": round(delta_a, 6),
        "condition_b_delta": round(delta_b, 6),
        "condition_c_delta": round(delta_c, 6),
        "max_head_layer":   top1_layer,
        "max_head_index":   top1_head,
    })

    if (row_idx + 1) % 10 == 0 or row_idx == 0:
        print(f"  [{row_idx+1:>3}/{len(df)}]  '{prompt}' → "
              f"baseline={p_baseline:.4f}  Δa={delta_a:+.3f}  "
              f"Δb={delta_b:+.3f}  Δc={delta_c:+.3f}  "
              f"top head=L{top1_layer}H{top1_head}")

print(f"\nDone — {len(results)} prompts processed.")

## 8. Save & Display Results

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)
print("Saved results.csv")
results_df

## 9. Summary Statistics

In [ ]:
print("=" * 65)
print("ABLATION IMPACT SUMMARY  (relative probability decrease)")
print("=" * 65)
for cond in ["condition_a_delta", "condition_b_delta", "condition_c_delta"]:
    vals = results_df[cond]
    print(f"  {cond:>20s}:  mean={vals.mean():.4f}  "
          f"median={vals.median():.4f}  max={vals.max():.4f}")
print()

# Most-targeted layer/head distribution
print("Most-influential head distribution (Condition A):")
head_counts = results_df.groupby(["max_head_layer", "max_head_index"]).size()
head_counts = head_counts.sort_values(ascending=False)
print(head_counts.head(10).to_string())
print("=" * 65)